# 01 — Data Understanding

This notebook checks the local Credit Card Fraud Detection dataset's structure, documented feature meanings, target balance, and basic data quality. It does not train models or perform advanced exploratory analysis.

Run from the repository root or from this notebook: the path resolution below searches upward for `data/raw/creditcard.csv`.

In [ ]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
from IPython.display import display

# Reproducibility: this inspection is deterministic; set a seed for any later
# randomized sampling or analysis added to this notebook.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

In [ ]:
def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw" / "creditcard.csv").is_file():
            return candidate
    raise FileNotFoundError("Could not find data/raw/creditcard.csv from the current directory or its parents.")

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "creditcard.csv"
df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Dataset and feature documentation

The repository's `data/README.md` identifies the Kaggle Credit Card Fraud Detection dataset. Its source description says transactions cover two days; `Time` is elapsed time in seconds from the first transaction, `Amount` is the transaction amount, and `V1` through `V28` are PCA principal components. The original attributes behind those components are not disclosed, so this notebook does not assign them individual business meanings. `Class` is the binary label: 1 indicates fraud and 0 indicates a non-fraud transaction.

Reference: [Kaggle dataset description](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud). The local CSV is inspected as-is; no source file is modified.

In [ ]:
print("Column names:")
print(df.columns.tolist())
print("\nDataFrame structure:")
df.info()

The documented meanings are limited for this dataset: `V1`–`V28` are transformed and anonymized. The original transaction attributes cannot be recovered from their names alone. `Time` and `Amount` retain their documented interpretations, while `Class` supplies the outcome label.

In [ ]:
dtype_summary = df.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("column_count")
display(dtype_summary)

missing = df.isna().sum().rename("missing_values").to_frame()
display(missing.T)

CSV parsing produces numeric columns in this file. The missing-value check reports parsed null values; an empty result supports basic completeness but does not rule out duplicate rows, invalid values, or other data quality issues. Those checks can be considered in later preparation work.

In [ ]:
target = df["Class"].value_counts(dropna=False).sort_index()
target_summary = pd.DataFrame({
    "count": target,
    "fraction": target / len(df),
})
display(target_summary)

fraud_rate = target.get(1, 0) / len(df)
majority_accuracy = target.max() / len(df)
print(f"Fraud prevalence: {fraud_rate:.4%}")
print(f"Majority-class-only accuracy: {majority_accuracy:.4%}")

The fraud class is rare. A rule that predicts the majority class for every row would have high apparent accuracy but zero fraud recall. This is why later evaluation should show fraud-specific precision and recall, and make false-positive and false-negative counts visible.

In [ ]:
display(df.describe().T)

These descriptive statistics summarize numeric distributions without interpreting the anonymized components or making modeling decisions. Values in `Time` and `Amount` are on different scales from the PCA components; any later preprocessing choice should be made with the eventual modeling method and train/validation split in mind.

In [ ]:
duplicate_rows = int(df.duplicated().sum())
invalid_target = sorted(set(df["Class"].dropna().unique()) - {0, 1})
print(f"Duplicate rows: {duplicate_rows:,}")
print(f"Unexpected target values outside {{0, 1}}: {invalid_target}")
print(f"Rows with any missing value: {int(df.isna().any(axis=1).sum()):,}")

## Initial observations and limitations

- The inspected file contains 284,807 transactions and 31 columns: 30 input columns plus `Class`.
- It contains 492 fraud labels (about 0.173%), so class imbalance is substantial.
- The file has no parsed missing values. Duplicate and target checks above provide additional basic checks when run.
- The feature names and dataset source do not expose original attributes for `V1`–`V28`; their individual meaning is unknown here.
- The dataset covers only two days in 2013 and should not be treated as representative of current production fraud without further evidence.

Before Phase 3, review the intended prediction point and operational response, choose a leakage-aware validation strategy, and clarify how false positives, false negatives, and review capacity should be weighed. Do not infer a production threshold from this dataset alone.